<a href="https://colab.research.google.com/github/giannisster/knn-and-centroid-tests-over-cifar10/blob/main/DL_Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pickle
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

In [ ]:
def load_batch(file_path):
    with open(file_path, 'rb') as f:
        data_dict = pickle.load(f, encoding='bytes')
    # data is a 10000x3072 numpy array of uint8.
    # Each row is a 1D representation of a 32x32 color image.
    X = data_dict[b'data']
    y = np.array(data_dict[b'labels'])
    return X, y

# List of training batch files
train_files = [
    '/content/data_batch_1',
   '/content/data_batch_2',
     '/content/data_batch_3',
     '/content/data_batch_4',
     '/content/data_batch_5'
]

# Load and concatenate all training batches
X_train_list, y_train_list = [], []
for file in train_files:
    X, y = load_batch(file)
    X_train_list.append(X)
    y_train_list.append(y)

X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

# Load test batch
X_test, y_test = load_batch('/content/test_batch')

# normalize data
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (50000, 3072)
y_train shape: (50000,)
X_test shape: (10000, 3072)
y_test shape: (10000,)


In [ ]:
#Implementing KNN from scratch

class MyKNN:

  def predict(self, x_train, y_train ,x_test ,k=10 ,distance='euclidian'):

    if distance == 'euclidian':
      distances = np.linalg.norm(x_train - x_test, axis=1)
    elif distance == 'manhattan':
      distances = np.sum(np.abs(x_train - x_test), axis=1)
    elif distance == 'chebyshev':
      distances = np.max(np.abs(x_train - x_test), axis=1)

    idx = np.argsort(distances)[:k]
    neighbours_labels = y_train[idx]
    y_pred = Counter(neighbours_labels).most_common(1)[0][0]
    return y_pred


In [ ]:
knn = MyKNN()
test_subset_size = 100 # limited subset cause it takes forever

# Test with kNN with k=1
for metric in ['euclidian', 'manhattan', 'chebyshev']:
    error = 0
    for i in range(test_subset_size):
        pred = knn.predict(X_train, y_train, X_test[i], k=1, distance=metric)
        error += 0 if pred == y_test[i] else 1
    print(f"error in test dataset using {metric} distance metric (k=1): {error/test_subset_size * 100:.2f}%")

error: 69.00%


In [ ]:
# Test with kNN with k=3
for metric in ['euclidian', 'manhattan', 'chebyshev']:
    error = 0
    for i in range(test_subset_size):
        pred = knn.predict(X_train, y_train, X_test[i], k=3, distance=metric)
        error += 0 if pred == y_test[i] else 1
    print(f"error in test dataset using {metric} distance metric (k=3): {error/test_subset_size * 100:.2f}%")

array([3, 8, 8, ..., 5, 1, 7])

In [ ]:
#Implementing Nearest Centroid from scratch

class MyNearestCentroid:

  def fit(self, x_train, y_train):

    self.classes = np.unique(y_train)
    self.centroids = np.array([
      np.mean(x_train[y_train == c], axis=0)
      for c in self.classes
      ])


  def predict(self, x, distance='euclidian'):

    if distance == 'euclidian':
      distances = np.linalg.norm(self.centroids - x, axis=1)
    elif distance == 'manhattan':
      distances = np.sum(np.abs(self.centroids - x), axis=1)
    elif distance == 'chebyshev':
      distances = np.max(np.abs(self.centroids - x), axis=1)

    idx = np.argmin(distances)
    neighbour_label = self.classes[idx]
    y_pred = neighbour_label
    return y_pred

In [ ]:
# Test Nearest Centroid with different distance metrics
nc = MyNearestCentroid()
nc.fit(X_train, y_train)
test_subset_size = 10000

for metric in ['euclidian', 'manhattan', 'chebyshev']:
    error = 0
    for i in range(test_subset_size):
        pred = nc.predict(X_test[i], distance=metric)
        error += 0 if pred == y_test[i] else 1
    print(f"error in test dataset using {metric} distance metric (Nearest Centroid): {error/test_subset_size * 100:.2f}%")

### Evaluating KNN (k=3) on the Training Dataset (Subset Size: 100)

In [ ]:
train_subset_size = 100

# Test KNN (k=3) on training subset
for metric in ['euclidian', 'manhattan', 'chebyshev']:
    error = 0
    for i in range(train_subset_size):
        # Predicting for training sample i using the rest of the training set
        # To prevent predicting a point using itself as its own neighbor, we can pass the training set directly
        # but kNN will trivially match itself if k=1. For k=3, the sample itself is still included.
        pred = knn.predict(X_train, y_train, X_train[i], k=3, distance=metric)
        error += 0 if pred == y_train[i] else 1
    print(f"Error in train dataset using {metric} distance metric (k=3): {error/train_subset_size * 100:.2f}%")

### Evaluating Nearest Centroid on the Training Dataset

In [ ]:
# Test Nearest Centroid on the full training dataset
train_subset_size_nc = 10000  # Evaluating on 10,000 samples of train dataset for reasonable runtime

for metric in ['euclidian', 'manhattan', 'chebyshev']:
    error = 0
    for i in range(train_subset_size_nc):
        pred = nc.predict(X_train[i], distance=metric)
        error += 0 if pred == y_train[i] else 1
    print(f"Error in train dataset using {metric} distance metric (Nearest Centroid): {error/train_subset_size_nc * 100:.2f}%")